In [1]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Added to sys.path:", repo_root)
from fixedincomelib import *
print("Fixed Income Library is loaded.")

Added to sys.path: c:\QuantBricker
Fixed Income Library is loaded.


### Build yield curve sub model

In [2]:
bm_list_yc = []
# create yield curve build method
content_sofr = {
    "TARGET": "SOFR-1B",
    "OVERNIGHT INDEX FUTURE": "SOFR-FUTURE-3M",
    "OVERNIGHT INDEX SWAP": "USD-SOFR-OIS",
}
bm_list_yc.append(qfCreateBuildMethod("YIELD_CURVE_INDEX", content_sofr))
# funding build method
content_sofr_funding = {
    "TARGET": "SOFR-1B-FLAT",
    "SPREAD ZERO RATE": "SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD",
}
bm_list_yc.append(qfCreateBuildMethod("YIELD_CURVE_FUNDING", content_sofr_funding))
# create yield curve common build method
content = {"TARGET": "USD", "FUNDING PARAMETERS": "USD-FUNDING-PARAMETERS", "SOLVER": "BRENTQ"}
bm_list_yc.append(qfCreateBuildMethod("YIELD_CURVE_COMMON", content))
build_method_collection_yc = qfCreateModelBuildMethodCollection(bm_list_yc)

In [3]:
### ois futures
df_fut = pd.DataFrame(
    [
        ["2026-03-19x2026-06-18", 96.44],
        ["2026-06-18x2026-09-17", 96.70],
        ["2026-09-17x2026-12-10", 96.85],
        ["2026-12-10x2027-03-17", 96.90],
        ["2027-03-17x2027-06-16", 96.91],
        ["2027-06-16x2027-09-15", 96.89],
        ["2027-09-15x2027-12-15", 96.85],
        ["2027-12-15x2028-03-15", 96.81],
        ["2028-03-15x2028-06-21", 96.76],
        ["2028-06-21x2028-09-20", 96.71],
        ["2028-09-20x2028-12-20", 96.66],
        ["2028-12-20x2029-03-21", 96.61],
    ],
    columns=["axis1", "values"],
).set_index("axis1")
data_fut = qfCreateData1D("OVERNIGHT INDEX FUTURE", "SOFR-FUTURE-3M", df_fut)

In [4]:
### ois swap
df_swap = pd.DataFrame(
    [
        ["4Y", 0.03358],
        ["5Y", 0.03422],
        ["6Y", 0.03491],
        ["7Y", 0.03560],
        ["8Y", 0.03624],
        ["9Y", 0.03685],
        ["10Y", 0.03742],
        ["12Y", 0.03849],
        ["15Y", 0.03974],
        ["20Y", 0.04087],
        ["25Y", 0.04110],
        ["30Y", 0.04089],
        ["35Y", 0.04044],
        ["40Y", 0.03996],
        ["50Y", 0.03887],
        ["60Y", 0.03772],
    ],
    columns=["axis1", "values"],
).set_index("axis1")
data_swap = qfCreateData1D("OVERNIGHT INDEX SWAP", "USD-SOFR-OIS", df_swap)

In [5]:
# spread zero rate
df_spread_zero_rate_rfr = pd.DataFrame(
    [["1Y", 0.0], ["5Y", 0.0], ["10Y", 0.0], ["20Y", 0.0], ["30Y", 0.0]],
    columns=["axis1", "values"],
).set_index("axis1")
data_szr_rfr = qfCreateData1D(
    "SPREAD ZERO RATE", "SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD", df_spread_zero_rate_rfr
)

In [6]:
### funding parameter table
df_dg = pd.DataFrame(
    [
        ["Overnight Index Future", "SOFR-FUTURE-3M", "SOFR-1B-FLAT"],
        ["Overnight Index Swap", "USD-SOFR-OIS", "SOFR-1B-FLAT"],
    ],
    columns=["DATA TYPE", "DATA CONVENTION", "FUNDING IDENTIFIER"],
)
data_fpt = qfCreateDataGeneric("DATA GENERIC", "USD-FUNDING-PARAMETERS", df_dg)

In [7]:
### pack up all data into data collection
data_collection_yc_list = [data_szr_rfr, data_fut, data_swap, data_fpt]
data_collection_yc = qfCreateDataCollection(data_collection_yc_list)

In [8]:
value_date = "2026-02-11"
yc_model = qfCreateModel(value_date, "YIELD_CURVE", data_collection_yc, build_method_collection_yc)
path = "serialized/yc_model_calibrated.pickle"
qfWriteModelObjectToFile(yc_model, path)

'DONE'

### Create Build Method Collection

In [9]:
build_method_type = 'IR_SABR'
content = {
    'TARGET' : 'SOFR-1B-SWAPTION',
    'SWAPTION NORMAL VOLATILITY' : 'USD-SOFR-SWAPTION',
    'SWAPTION SABR BETA' : 'USD-SOFR-SWAPTION',
    'SWAPTION SABR NU' : 'USD-SOFR-SWAPTION',
    'SWAPTION SABR RHO' : 'USD-SOFR-SWAPTION',
    'VOL INTERPOLATION DOMAIN' : 'NORMAL VOLATILITY',
    'INTERPOLATION METHOD' : 'LINEAR',
    'EXTRAPOLATION METHOD' : 'FLAT',
    'BUSINESSDAY CONVENTION' : 'F',
    'HOLIDAY CONVENTION' : 'USGS',
    'SHIFT' : 0.04
}
sabr_build_method = qfCreateBuildMethod(build_method_type, content)
display(sabr_build_method.display())

,Name,Value
0,TARGET,SOFR-1B-SWAPTION
1,SWAPTION NORMAL VOLATILITY,USD-SOFR-SWAPTION
2,SWAPTION SABR BETA,USD-SOFR-SWAPTION
3,SWAPTION SABR NU,USD-SOFR-SWAPTION
4,SWAPTION SABR RHO,USD-SOFR-SWAPTION
5,VOL INTERPOLATION DOMAIN,NORMAL VOLATILITY
6,INTERPOLATION METHOD,LINEAR
7,EXTRAPOLATION METHOD,FLAT
8,BUSINESSDAY CONVENTION,F
9,HOLIDAY CONVENTION,USGS


In [10]:
### serialize / de-serialize
path = 'sabr_build_method.pickle'
qfWriteBuildMethodToFile(sabr_build_method, path)
display(sabr_build_method.display())
sabr_build_method_back = qfReadBuildMethodFromFile(path)
# check
display(sabr_build_method_back.display())
# house keeping
os.remove(path)

,Name,Value
0,TARGET,SOFR-1B-SWAPTION
1,SWAPTION NORMAL VOLATILITY,USD-SOFR-SWAPTION
2,SWAPTION SABR BETA,USD-SOFR-SWAPTION
3,SWAPTION SABR NU,USD-SOFR-SWAPTION
4,SWAPTION SABR RHO,USD-SOFR-SWAPTION
5,VOL INTERPOLATION DOMAIN,NORMAL VOLATILITY
6,INTERPOLATION METHOD,LINEAR
7,EXTRAPOLATION METHOD,FLAT
8,BUSINESSDAY CONVENTION,F
9,HOLIDAY CONVENTION,USGS


,Name,Value
0,HOLIDAY CONVENTION,USGS
1,VOL INTERPOLATION DOMAIN,NORMAL VOLATILITY
2,INTERPOLATION METHOD,LINEAR
3,EXTRAPOLATION METHOD,FLAT
4,SHIFT,0.04
5,SWAPTION SABR NU,USD-SOFR-SWAPTION
6,SWAPTION NORMAL VOLATILITY,USD-SOFR-SWAPTION
7,SWAPTION SABR BETA,USD-SOFR-SWAPTION
8,SWAPTION SABR RHO,USD-SOFR-SWAPTION
9,BUSINESSDAY CONVENTION,F


In [11]:
### caplet
build_method_type = 'IR_SABR'
content_cf = {
    'TARGET' : 'SOFR-1B-CAPFLOOR',
    'SWAPTION NORMAL VOLATILITY' : 'USD-SOFR-CAPFLOOR',
    'SWAPTION SABR BETA' : 'USD-SOFR-CAPFLOOR',
    'SWAPTION SABR NU' : 'USD-SOFR-CAPFLOOR',
    'SWAPTION SABR RHO' : 'USD-SOFR-CAPFLOOR',
    'VOL INTERPOLATION DOMAIN' : 'NORMAL VOLATILITY',
    'INTERPOLATION METHOD' : 'LINEAR',
    'EXTRAPOLATION METHOD' : 'FLAT',
    'BUSINESSDAY CONVENTION' : 'F',
    'HOLIDAY CONVENTION' : 'USGS',
    'SHIFT' : 0.04
}
sabr_build_method_cf = qfCreateBuildMethod(build_method_type, content_cf)
display(sabr_build_method_cf.display())

,Name,Value
0,TARGET,SOFR-1B-CAPFLOOR
1,SWAPTION NORMAL VOLATILITY,USD-SOFR-CAPFLOOR
2,SWAPTION SABR BETA,USD-SOFR-CAPFLOOR
3,SWAPTION SABR NU,USD-SOFR-CAPFLOOR
4,SWAPTION SABR RHO,USD-SOFR-CAPFLOOR
5,VOL INTERPOLATION DOMAIN,NORMAL VOLATILITY
6,INTERPOLATION METHOD,LINEAR
7,EXTRAPOLATION METHOD,FLAT
8,BUSINESSDAY CONVENTION,F
9,HOLIDAY CONVENTION,USGS


In [12]:
### pack up as build method collection
bm_list = [sabr_build_method, sabr_build_method_cf]
bm_collection = qfCreateModelBuildMethodCollection(bm_list)
bm_collection.display()

,Name,Value
0,IR_SABR,SOFR-1B-SWAPTION
1,IR_SABR,SOFR-1B-CAPFLOOR


In [13]:
bm_list_merged = bm_list + bm_list_yc
bm_collection_merged = qfCreateModelBuildMethodCollection(bm_list_merged)
bm_collection_merged.display()

,Name,Value
0,IR_SABR,SOFR-1B-SWAPTION
1,IR_SABR,SOFR-1B-CAPFLOOR
2,YIELD_CURVE_INDEX,SOFR-1B
3,YIELD_CURVE_FUNDING,SOFR-1B-FLAT
4,YIELD_CURVE_COMMON,USD


### Data

In [14]:
### swaption
expiries = ['3M', '6M', '1Y', '5Y', '10Y']
tenors = ['1Y', '5Y', '10Y']
data_conv = 'USD-SOFR-SWAPTION'
# nv
df = pd.DataFrame(np.random.uniform(low=90./1e4, high=110./1e4, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_nv = qfCreateData2D('Swaption Normal Volatility', data_conv, df)
# beta
df = pd.DataFrame(np.random.uniform(low=60./1e2, high=60./1e2, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_beta = qfCreateData2D('Swaption SABR Beta', data_conv, df)
# nu
df = pd.DataFrame(np.random.uniform(low=0.01, high=0.3, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_nu= qfCreateData2D('Swaption SABR Nu', data_conv, df)
# rho
df = pd.DataFrame(np.random.uniform(low=-0.99, high=0.99, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_rho = qfCreateData2D('Swaption SABR Rho', data_conv, df)

In [15]:
### capfloor
expiries = ['3M', '6M', '1Y', '5Y', '10Y']
tenors = ['1Y', '5Y', '10Y']
data_conv = 'USD-SOFR-CAPFLOOR'
# nv
df = pd.DataFrame(np.random.uniform(low=90./1e4, high=110./1e4, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_nv_cf = qfCreateData2D('Swaption Normal Volatility', data_conv, df)
# beta
df = pd.DataFrame(np.random.uniform(low=60./1e2, high=60./1e2, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_beta_cf = qfCreateData2D('Swaption SABR Beta', data_conv, df)
# nu
df = pd.DataFrame(np.random.uniform(low=0.01, high=0.3, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_nu_cf= qfCreateData2D('Swaption SABR Nu', data_conv, df)
# rho
df = pd.DataFrame(np.random.uniform(low=-0.99, high=0.99, size=(len(expiries), len(tenors))), index=expiries, columns=tenors)
data2D_rho_cf = qfCreateData2D('Swaption SABR Rho', data_conv, df)

In [16]:
### create a data collection
data_list = [
    data2D_nv, data2D_beta, data2D_nu, data2D_rho,
    data2D_nv_cf, data2D_beta_cf, data2D_nu_cf, data2D_rho_cf]
data_collection = qfCreateDataCollection(data_list)
display(data_collection.display())

,Data Shape,Data Type,Data Convention
0,DATA2D,Swaption Normal Volatility,USD-SOFR-SWAPTION
1,DATA2D,Swaption SABR Beta,USD-SOFR-SWAPTION
2,DATA2D,Swaption SABR Nu,USD-SOFR-SWAPTION
3,DATA2D,Swaption SABR Rho,USD-SOFR-SWAPTION
4,DATA2D,Swaption Normal Volatility,USD-SOFR-CAPFLOOR
5,DATA2D,Swaption SABR Beta,USD-SOFR-CAPFLOOR
6,DATA2D,Swaption SABR Nu,USD-SOFR-CAPFLOOR
7,DATA2D,Swaption SABR Rho,USD-SOFR-CAPFLOOR


In [17]:
data_collection_merged_list = data_list + data_collection_yc_list
data_collection_merged = qfCreateDataCollection(data_collection_merged_list)
display(data_collection_merged .display())

,Data Shape,Data Type,Data Convention
0,DATA2D,Swaption Normal Volatility,USD-SOFR-SWAPTION
1,DATA2D,Swaption SABR Beta,USD-SOFR-SWAPTION
2,DATA2D,Swaption SABR Nu,USD-SOFR-SWAPTION
3,DATA2D,Swaption SABR Rho,USD-SOFR-SWAPTION
4,DATA2D,Swaption Normal Volatility,USD-SOFR-CAPFLOOR
5,DATA2D,Swaption SABR Beta,USD-SOFR-CAPFLOOR
6,DATA2D,Swaption SABR Nu,USD-SOFR-CAPFLOOR
7,DATA2D,Swaption SABR Rho,USD-SOFR-CAPFLOOR
8,DATA1D,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD
9,DATA1D,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M


### create sabr model and pricing

In [18]:
sabr_model = qfCreateSABRModel(
    sub_model=yc_model,
    data_collection=data_collection,
    build_method_collection=bm_collection
)
print("SABR components:", sabr_model.components_.keys())

SABR components: dict_keys(['SOFR-1B-SWAPTION', 'SOFR-1B-CAPFLOOR'])


In [19]:
sabr_model = qfCreateModel(
    value_date,
    'IR_SABR',
    data_collection=data_collection_merged,
    build_method_collection=bm_collection_merged
)

In [20]:
sabr_model = SABRModelBuilder.create_sabr_model(
    sub_model=yc_model,
    data_collection=data_collection,
    build_method_collection=bm_collection
)

print("SABR components:", sabr_model.components_.keys())

SABR components: dict_keys(['SOFR-1B-SWAPTION', 'SOFR-1B-CAPFLOOR'])


In [21]:
params = sabr_model.get_sabr_parameters(
    IndexRegistry().get("SOFR-1B-CAPFLOOR"),
    0.5, 
    0.25 
)

print("interpolated SABR parameters:")
for k, v in params.items():
    print(k, v)

interpolated SABR parameters:
SABRParameters.RHO -0.7635843497357558
SABRParameters.NU 0.06077056872731679
SABRParameters.NV 0.01067020433861178
SABRParameters.BETA 0.6


In [22]:
caplet = qfCreateProductRFRCapletFloorlet(
    effective_date="2026-06-17",
    expiry_offset="0D",
    term_or_termination_date="3M",
    payment_date="2026-09-21",
    on_index="SOFR-1B",
    strike=0.045,
    cap_or_floor="CAP",
    notional=1_000_000,
    accrual_basis="ACT/360",
    long_or_short="LONG",
)

qfDisplayProduct(caplet)

,Name,Value
0,Product Type,PRODUCT_RFR_CAPLET_FLOORLET
1,Notional,1000000
2,Currency,USD
3,Long Or Short,LONG
4,Effective Date,2026-06-17
5,Expiry Offset,0D
6,Expiry Date,2026-06-17
7,Termination Date,2026-09-17
8,Payment Date,2026-09-21
9,ON Index,SOFRON Actual/360


In [23]:
vp_funding = FundingIndexParameter({"Funding Index": "SOFR-1B-FLAT"})
vpc = ValuationParametersCollection([vp_funding])

request = ValuationRequest.PV

In [24]:
ve = ValuationEngineRFRCapletFloorlet(
    sabr_model,
    vpc,
    caplet,
    request,
)

ve.calculate_value()

print("PV           :", ve.value_)
print("Cash         :", ve.cash_)
print("DF           :", ve.df_)
print("Forward      :", ve.forward_)
print("Option Value :", ve.option_value_)
print("Expiry Date  :", ve.expiry_date_)
print("Tenor        :", ve.tenor_)

PV           : 15.593350272292097
Cash         : 0.0
DF           : 0.979063765902685
Forward      : 0.03302976270161267
Option Value : 6.232225081986625e-05
Expiry Date  : June 17th, 2026
Tenor        : 0.25555555555555554


In [25]:
grad = []
ve.calculate_first_order_risk(grad)

for i, g in enumerate(grad):
    print(f"component {i}: shape={g.shape}")
    print(g)

component 0: shape=(5,)
[0. 0. 0. 0. 0.]
component 1: shape=(28,)
[  76.40738486 6953.07202182    0.            0.            0.
    0.            0.            0.            0.            0.
    0.            0.            0.            0.            0.
    0.            0.            0.            0.            0.
    0.            0.            0.            0.            0.
    0.            0.            0.        ]
component 2: shape=(60,)
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
component 3: shape=(60,)
[ 5.62201118e+03  0.00000000e+00  0.00000000e+00  3.78208025e+03
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  4.38225677e+00
  0.00000000e+00  0.00000000e+00  2.94806365e+00  0.00000000e+00
  0.00000000e+00

In [26]:
cf_report = ve.create_cash_flows_report()
pv_cash_report = ve.get_value_and_cash()

display(cf_report.display())
display(pv_cash_report.display())

,PRODUCT_TYPE,VALUATION_ENGINE_TYPE,LEG_ID,CASHFLOW_ID,PAY_OR_RECEIVE,NOTIONAL,PAY_DATE,FORECASTED_AMOUNT,PV,DISCOUNG FACTOR,FIXING_DATE,START_DATE,END_DATE,ACCRUED,INDEX_OR_FIXED,INDEX_VALUE
0,PRODUCT_RFR_CAPLET_FLOORLET,ValuationEngineRFRCapletFloorlet,0,0,1.0,1000000,"September 21st, 2026",15.926797,15.59335,0.979064,"June 17th, 2026","June 17th, 2026","September 17th, 2026",0.255556,SOFRON Actual/360,0.03303


,Currency,Type,Value
0,USD,PV,15.59335
1,USD,CASH,0.00000


In [27]:
### test risk
df_risk = qfCreateValueReport(sabr_model, caplet, vpc, 'firstorderrisk').display()
df_risk

,DATA_TYPE,DATA_CONVENTION,AXIS1,AXIS2,MARKET_QUOTE,UNIT,VALUES
0,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,1Y,,0.0,0.0001,0.0
1,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,5Y,,0.0,0.0001,0.0
2,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,10Y,,0.0,0.0001,0.0
3,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,20Y,,0.0,0.0001,0.0
4,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,30Y,,0.0,0.0001,0.0
...,...,...,...,...,...,...,...
148,SWAPTION SABR RHO,USD-SOFR-CAPFLOOR,5Y,5Y,-0.953489,1.0,0.0
149,SWAPTION SABR RHO,USD-SOFR-CAPFLOOR,5Y,10Y,-0.090973,1.0,0.0
150,SWAPTION SABR RHO,USD-SOFR-CAPFLOOR,10Y,1Y,0.083402,1.0,0.0
151,SWAPTION SABR RHO,USD-SOFR-CAPFLOOR,10Y,5Y,0.806668,1.0,0.0


In [28]:
df_risk[np.abs(df_risk["VALUES"].astype(float)) > 1e-10]

,DATA_TYPE,DATA_CONVENTION,AXIS1,AXIS2,MARKET_QUOTE,UNIT,VALUES
5,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-03-19x2026-06-18,,96.44,-0.01,0.007678
6,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-06-18x2026-09-17,,96.7,-0.01,0.699132
93,SWAPTION NORMAL VOLATILITY,USD-SOFR-CAPFLOOR,3M,1Y,0.009836,1.0,5622.011178
96,SWAPTION NORMAL VOLATILITY,USD-SOFR-CAPFLOOR,6M,1Y,0.010671,1.0,3782.080247
108,SWAPTION SABR BETA,USD-SOFR-CAPFLOOR,3M,1Y,0.6,1.0,4.382257
111,SWAPTION SABR BETA,USD-SOFR-CAPFLOOR,6M,1Y,0.6,1.0,2.948064
123,SWAPTION SABR NU,USD-SOFR-CAPFLOOR,3M,1Y,0.185354,1.0,-3.488584
126,SWAPTION SABR NU,USD-SOFR-CAPFLOOR,6M,1Y,0.060664,1.0,-2.346866
138,SWAPTION SABR RHO,USD-SOFR-CAPFLOOR,3M,1Y,0.169353,1.0,4.453588
141,SWAPTION SABR RHO,USD-SOFR-CAPFLOOR,6M,1Y,-0.767853,1.0,2.996050


In [29]:
print("TTE:", ve.time_to_expiry_)
print("Forward:", ve.forward_)
print("Strike:", ve.strike_)
print("NV/BETA/NU/RHO:", ve.sabr_result_.get(SABRParameters.NV), ve.sabr_result_.get(SABRParameters.BETA), ve.sabr_result_.get(SABRParameters.NU), ve.sabr_result_.get(SABRParameters.RHO))
print("Option Value:", ve.option_value_)

TTE: 0.3452054794520548
Forward: 0.03302976270161267
Strike: 0.045
NV/BETA/NU/RHO: 0.010172030778473272 0.6 0.13520700354945467 -0.20756719338389779
Option Value: 6.232225081986625e-05


### Bump Reval

In [30]:
pv_base = qfCreateValueReport(sabr_model, caplet, vpc, "pv")[0][1]
print(f"Base PV: {pv_base}")

### inspect non-zero SABR risks
df_risk_nonzero = df_risk[np.abs(df_risk["VALUES"].astype(float)) > 1e-10].copy()
display(df_risk_nonzero)

Base PV: 15.593350272292097


,DATA_TYPE,DATA_CONVENTION,AXIS1,AXIS2,MARKET_QUOTE,UNIT,VALUES
5,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-03-19x2026-06-18,,96.44,-0.01,0.007678
6,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-06-18x2026-09-17,,96.7,-0.01,0.699132
93,SWAPTION NORMAL VOLATILITY,USD-SOFR-CAPFLOOR,3M,1Y,0.009836,1.0,5622.011178
96,SWAPTION NORMAL VOLATILITY,USD-SOFR-CAPFLOOR,6M,1Y,0.010671,1.0,3782.080247
108,SWAPTION SABR BETA,USD-SOFR-CAPFLOOR,3M,1Y,0.6,1.0,4.382257
111,SWAPTION SABR BETA,USD-SOFR-CAPFLOOR,6M,1Y,0.6,1.0,2.948064
123,SWAPTION SABR NU,USD-SOFR-CAPFLOOR,3M,1Y,0.185354,1.0,-3.488584
126,SWAPTION SABR NU,USD-SOFR-CAPFLOOR,6M,1Y,0.060664,1.0,-2.346866
138,SWAPTION SABR RHO,USD-SOFR-CAPFLOOR,3M,1Y,0.169353,1.0,4.453588
141,SWAPTION SABR RHO,USD-SOFR-CAPFLOOR,6M,1Y,-0.767853,1.0,2.996050


In [39]:
risk_data_type = "SWAPTION NORMAL VOLATILITY"
risk_data_convention = "USD-SOFR-CAPFLOOR"
risk_expiry = "3M"
risk_tenor = "1Y"

bump_size = 1e-4

In [38]:
data2D_nv_cf.display()

,1Y,5Y,10Y
3M,0.009836,0.009731,0.010705
6M,0.010671,0.010290,0.010515
1Y,0.010561,0.010541,0.010060
5Y,0.010743,0.009460,0.010126
10Y,0.010023,0.010256,0.009820


In [41]:
# Step 1: build bumped up model
df_nv_cf_bumped = data2D_nv_cf.display().copy()
cur_nv = df_nv_cf_bumped.loc[risk_expiry, risk_tenor]
df_nv_cf_bumped.loc[risk_expiry, risk_tenor] = cur_nv + bump_size
data2D_nv_cf_bumped = qfCreateData2D(risk_data_type, risk_data_convention, df_nv_cf_bumped)
data_list_bumped = [data2D_nv, data2D_beta, data2D_nu, data2D_rho,
                    data2D_nv_cf_bumped, data2D_beta_cf, data2D_nu_cf, data2D_rho_cf]
data_collection_bumped = qfCreateDataCollection(data_list_bumped)
sabr_model_bumped = qfCreateSABRModel(sub_model=yc_model,
                                      data_collection= data_collection_bumped,
                                      build_method_collection=bm_collection)

# Step 2: re-value the same product
pv_bumped = qfCreateValueReport(sabr_model_bumped, caplet, vpc, 'pv')[0][1]

# Step 3 : bump-reval risk
risk_bump_reval = pv_bumped - pv_base
print(f'Bump reval risk is {risk_bump_reval:.10f}.')

# Step 4: analytic risk
analytic_risk = df_risk[
    (df_risk["DATA_TYPE"] == risk_data_type)
    & (df_risk["DATA_CONVENTION"] == risk_data_convention)
    & (df_risk["AXIS1"] == risk_expiry)
    & (df_risk["AXIS2"] == risk_tenor)
]['VALUES'].values[0]
analytic_scaled = analytic_risk * bump_size
print(f'Analytic risk is {analytic_scaled}')
print(f"Difference is {risk_bump_reval - analytic_scaled:.10e}.")

Bump reval risk is 0.5684639064.
Analytic risk is 0.5622011177597163
Difference is 6.2627886285e-03.
